# Flight Delay Regression Training

This notebook trains regression models to predict delay minutes using the labeled dataset produced by the data pipeline.
It follows the modeling requirements: chronological split, leakage-safe preprocessing, baseline comparison, and model artifacts.

- Dropped `temperature_c` to avoid multicollinearity.
- Added `class_weight='balanced'` to Two-Stage Classifier.
- TransformedTargetRegressor with `np.log1p` applied to Regressors.
- Added Permutation Feature Importance for Two-Stage Model.
- Used TimeSeriesSplit for Optuna tuning.

In [1]:
from __future__ import annotations

from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings("ignore")

In [2]:
data_path = Path("..") / "Data Collection + Processing" / "data" / "processed" / "training_dataset_labeled.csv"
df = pd.read_csv(data_path)
df.head()

,retrieved_at_vn,flight_date,direction,scheduled_time,estimated_time,route_airport,flight_number,status_raw,source_airport,route_airport_std,...,scheduled_month,sin_hour,cos_hour,airline_code,flight_num_only,minutes_to_departure_at_snapshot,is_estimated_missing,visibility_bin,temp_dew_spread,is_low_visibility
0,2026-04-17 22:01:45,2026-04-17,Arrival,22:15,22:23,TP. Hồ Chí Minh (SGN),VN148,Đúng giờ,DN,HO CHI MINH (SGN),...,4,-0.500000,0.866025,VN,148,13.250000,0,Medium,2,0.0
1,2026-04-18 00:10:37,2026-04-18,Departure,07:50,07:50,Hà Nội (HAN),VN158,3-16,DN,HA NOI (HAN),...,4,0.965926,-0.258819,VN,158,459.383333,0,Medium,3,0.0
2,2026-04-18 00:10:37,2026-04-18,Departure,08:50,08:50,Hà Nội (HAN),VN160,3-16,DN,HA NOI (HAN),...,4,0.866025,-0.500000,VN,160,519.383333,0,Medium,3,0.0
3,2026-04-18 00:10:37,2026-04-18,Departure,09:45,09:45,Hà Nội (HAN),VN164,3-16,DN,HA NOI (HAN),...,4,0.707107,-0.707107,VN,164,574.383333,0,Medium,3,0.0
4,2026-04-18 00:10:37,2026-04-18,Departure,11:35,11:35,Hà Nội (HAN),VN166,3-16,DN,HA NOI (HAN),...,4,0.258819,-0.965926,VN,166,684.383333,0,Medium,3,0.0


## Basic cleanup, Target capping, and Fix Data Leakage
- Parse `scheduled_dt` and sort for chronological split.
- Cap `delay_minutes` to [0, 300] to reduce extreme noise.

In [3]:
df["scheduled_dt"] = pd.to_datetime(df["scheduled_dt"], errors="coerce")
df = df.dropna(subset=["scheduled_dt", "delay_minutes"]).copy()

df = df.sort_values("scheduled_dt").reset_index(drop=True)

df["delay_minutes_raw"] = df["delay_minutes"]
df["delay_minutes"] = df["delay_minutes"].clip(lower=0, upper=300)

df[["delay_minutes_raw", "delay_minutes"]].describe()

,delay_minutes_raw,delay_minutes
count,1829.000000,1829.000000
mean,-0.570257,1.344997
std,20.162174,10.992570
min,-345.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.000000
max,245.000000,245.000000


## Feature Engineering & Selection
- **Multicollinearity**: Keep `temp_dew_spread`, drop `temperature_c`.

In [4]:
feature_cols = [
    "scheduled_hour", # Kept for tree-models, though sin/cos is also available
    "sin_hour",
    "cos_hour",
    "scheduled_dayofweek",
    "scheduled_month",
    "minutes_to_departure_at_snapshot",
    "source_airport",
    "direction",
    "route_airport_std",
    "flight_number",
    "airline_code",
    "flight_num_only",
    "is_estimated_missing",
    "dew_point_c",
    "wind_direction_deg",
    "wind_speed_kt",
    "visibility_miles",
    "visibility_bin",
    "cloud_cover",
    "temp_dew_spread",
    "is_wind_variable",
]

missing_cols = [c for c in feature_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

X = df[feature_cols].copy()
y = df["delay_minutes"].copy()

X.head()

,scheduled_hour,sin_hour,cos_hour,scheduled_dayofweek,scheduled_month,minutes_to_departure_at_snapshot,source_airport,direction,route_airport_std,flight_number,...,flight_num_only,is_estimated_missing,dew_point_c,wind_direction_deg,wind_speed_kt,visibility_miles,visibility_bin,cloud_cover,temp_dew_spread,is_wind_variable
0,22,-0.500000,8.660254e-01,4,4,13.250000,DN,Arrival,HO CHI MINH (SGN),VN148,...,148,0,25,130.0,7,6.00,Medium,BKN@800ft,2,0
1,5,0.965926,2.588190e-01,5,4,26.400000,NB,Arrival,HO CHI MINH,3S621,...,3,0,23,80.0,5,4.97,Medium,"BKN@2600ft, BKN@3000ft",1,0
2,5,0.965926,2.588190e-01,5,4,324.383333,DN,Departure,HO CHI MINH (SGN),VJ621,...,621,0,24,130.0,7,6.00,Medium,BKN@900ft,3,0
3,5,0.965926,2.588190e-01,5,4,334.383333,DN,Departure,HO CHI MINH (SGN),VN101,...,101,0,24,130.0,7,6.00,Medium,BKN@900ft,3,0
4,6,1.000000,6.123234e-17,5,4,349.383333,DN,Departure,BUON ME THUOT (BMV),VN1911,...,1911,0,24,130.0,7,6.00,Medium,BKN@900ft,3,0


## Chronological split
Using row-index based chronological split since dates might be sparse after filtering out data leakage.

In [5]:
n = len(df)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

print((X_train.shape, X_val.shape, X_test.shape))

((1280, 21), (274, 21), (275, 21))


## Preprocessing pipeline

In [6]:
categorical_cols = [
    "source_airport",
    "direction",
    "route_airport_std",
    "flight_number",
    "airline_code",
    "cloud_cover",
    "visibility_bin",
]

numeric_cols = [c for c in feature_cols if c not in categorical_cols]

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", ohe),
    ]
)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_cols),
        ("num", numeric_transformer, numeric_cols),
    ],
    remainder="drop",
)

## Baseline model

In [7]:
def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {"mae": mae, "rmse": rmse, "r2": r2}

baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_val)
baseline_metrics = evaluate_regression(y_val, baseline_pred)
print("Baseline Metrics:", baseline_metrics)

Baseline Metrics: {'mae': 1.686131386861314, 'rmse': 10.311569577580547, 'r2': -0.027472844302529387}


## Candidate models

In [8]:
# Wrap Regressors with Log1p Transform to handle right-skewed target
def get_log_regressor(regressor):
    return TransformedTargetRegressor(
        regressor=regressor,
        func=np.log1p,
        inverse_func=np.expm1,
    )

models = {
    "RandomForest": get_log_regressor(RandomForestRegressor(
        n_estimators=100, # Reduced for faster training
        max_depth=6,
        min_samples_split=4,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    ))
}

try:
    from sklearn.ensemble import HistGradientBoostingRegressor
    models["HistGB_Log1p"] = get_log_regressor(HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=6,
        max_iter=100,
        random_state=42,
    ))
except Exception:
    pass

try:
    from xgboost import XGBRegressor
    models["XGBoost"] = get_log_regressor(XGBRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
    ))
except Exception:
    pass

try:
    from lightgbm import LGBMRegressor
    models["LightGBM"] = get_log_regressor(LGBMRegressor(
        n_estimators=100,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1,
    ))
except Exception:
    pass

try:
    from catboost import CatBoostRegressor
    models["CatBoost"] = get_log_regressor(CatBoostRegressor(
        depth=6,
        learning_rate=0.08,
        iterations=100,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
    ))
except Exception:
    pass

print("Models:", models.keys())

Models: dict_keys(['RandomForest', 'HistGB_Log1p', 'XGBoost', 'LightGBM', 'CatBoost'])


In [9]:
results = []
trained = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocess", preprocess),
        ("model", model),
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)
    metrics = evaluate_regression(y_val, pred)
    metrics["model"] = name
    metrics["baseline_mae"] = baseline_metrics["mae"]
    results.append(metrics)
    trained[name] = pipe

results_df = pd.DataFrame(results).sort_values("mae")
print(results_df)

        mae      rmse        r2         model  baseline_mae
2  1.547198  9.178950  0.185846       XGBoost      1.686131
4  1.621983  9.446825  0.137632      CatBoost      1.686131
0  1.680899  9.770328  0.077558  RandomForest      1.686131
3  1.737889  9.944956  0.044289      LightGBM      1.686131
1  1.739805  9.993391  0.034957  HistGB_Log1p      1.686131


## Two-stage model (delay probability + magnitude)
Uses `class_weight='balanced'` for Classifier to address 85/15 imbalance.
Uses `TransformedTargetRegressor` with `np.log1p` for Regressor.

In [10]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier

def fit_two_stage(X_train, y_train, X_val, y_val):
    y_train_pos = (y_train > 0).astype(int)

    # Stage 1: Classifier with class_weight='balanced'
    clf = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=6,
        max_iter=100,
        random_state=42,
        class_weight='balanced'
    )
    clf_pipe = Pipeline([
        ("preprocess", clone(preprocess)),
        ("model", clf),
    ])
    clf_pipe.fit(X_train, y_train_pos)

    # Stage 2: Regressor with Log1p Transform
    base_reg = HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=6,
        max_iter=100,
        random_state=42,
    )
    reg = TransformedTargetRegressor(
        regressor=base_reg,
        func=np.log1p,
        inverse_func=np.expm1,
    )
    reg_pipe = Pipeline([
        ("preprocess", clone(preprocess)),
        ("model", reg),
    ])
    
    pos_mask = y_train > 0
    if pos_mask.sum() > 0:
        reg_pipe.fit(X_train[pos_mask], y_train[pos_mask])
    else:
        # Fallback if no positive cases
        reg_pipe.fit(X_train, y_train)

    val_prob = clf_pipe.predict_proba(X_val)[:, 1]
    val_reg = reg_pipe.predict(X_val)

    thresholds = np.linspace(0.1, 0.9, 17)
    best_threshold = 0.5
    best_mae = float("inf")
    best_pred = None
    for threshold in thresholds:
        val_pred = np.where(val_prob >= threshold, val_reg, 0.0)
        mae = mean_absolute_error(y_val, val_pred)
        if mae < best_mae:
            best_mae = mae
            best_threshold = threshold
            best_pred = val_pred

    metrics = evaluate_regression(y_val, best_pred)
    metrics["model"] = "TwoStage"
    metrics["baseline_mae"] = baseline_metrics["mae"]
    return clf_pipe, reg_pipe, best_threshold, metrics

two_stage_clf, two_stage_reg, two_stage_threshold, two_stage_metrics = fit_two_stage(
    X_train, y_train, X_val, y_val
)
print("Two Stage Metrics:", two_stage_metrics)

Two Stage Metrics: {'mae': 1.562460846404678, 'rmse': 9.718125498262264, 'r2': 0.08738877603959061, 'model': 'TwoStage', 'baseline_mae': 1.686131386861314}


## Select best model and evaluate on test

In [11]:
best_model_name = results_df.iloc[0]["model"]
best_pipe = trained[best_model_name]
best_mae = results_df.iloc[0]["mae"]

if "two_stage_metrics" in globals() and two_stage_metrics["mae"] < best_mae:
    best_model_name = "TwoStage"
    best_mae = two_stage_metrics["mae"]

X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

if best_model_name == "TwoStage":
    y_trainval_pos = (y_trainval > 0).astype(int)
    two_stage_clf.fit(X_trainval, y_trainval_pos)
    if (y_trainval > 0).sum() > 0:
        two_stage_reg.fit(X_trainval[y_trainval > 0], y_trainval[y_trainval > 0])
    else:
        two_stage_reg.fit(X_trainval, y_trainval)
        
    test_prob = two_stage_clf.predict_proba(X_test)[:, 1]
    test_reg = two_stage_reg.predict(X_test)
    test_pred = np.where(test_prob >= two_stage_threshold, test_reg, 0.0)
    test_metrics = evaluate_regression(y_test, test_pred)
else:
    best_pipe.fit(X_trainval, y_trainval)
    test_pred = best_pipe.predict(X_test)
    test_metrics = evaluate_regression(y_test, test_pred)

print("Best Model:", best_model_name)
print("Test Metrics:", test_metrics)

Best Model: XGBoost
Test Metrics: {'mae': 0.4182265006412159, 'rmse': 2.7206226349760265, 'r2': 0.009974918027931934}


## Feature importance (Permutation Importance)

In [12]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns

def get_feature_names(preprocess, categorical_cols, numeric_cols):
    cat_features = preprocess.named_transformers_["cat"].get_feature_names_out(categorical_cols)
    return np.concatenate([cat_features, numeric_cols])

if best_model_name == "TwoStage":
    print("Computing Permutation Feature Importance for Two-Stage Classifier...")
    y_val_pos = (y_val > 0).astype(int)
    # Use neg_brier_score or accuracy as scoring, by default permutation_importance uses the estimator's default score
    result = permutation_importance(
        two_stage_clf, X_val, y_val_pos, n_repeats=5, random_state=42, n_jobs=-1
    )
    
    importances = pd.Series(result.importances_mean, index=X_val.columns)
    top_importances = importances.sort_values(ascending=False).head(20)
    print("Top Importances:")
    print(top_importances)
else:
    print("Computing Permutation Feature Importance...")
    result = permutation_importance(
        best_pipe, X_val, y_val, n_repeats=5, random_state=42, n_jobs=-1
    )
    importances = pd.Series(result.importances_mean, index=X_val.columns)
    top_importances = importances.sort_values(ascending=False).head(20)
    print("Top Importances:")
    print(top_importances)

Computing Permutation Feature Importance...
Top Importances:
flight_number                       0.519155
direction                           0.190066
cloud_cover                         0.184094
temp_dew_spread                     0.090485
minutes_to_departure_at_snapshot    0.086108
source_airport                      0.039774
flight_num_only                     0.027578
airline_code                        0.026857
wind_direction_deg                  0.020894
scheduled_hour                      0.017184
wind_speed_kt                       0.013871
sin_hour                            0.010278
scheduled_dayofweek                 0.009491
visibility_miles                    0.002781
cos_hour                            0.001702
dew_point_c                         0.001259
route_airport_std                   0.000890
is_estimated_missing                0.000000
scheduled_month                     0.000000
visibility_bin                      0.000000
dtype: float64


## Save best model artifact

In [13]:
import joblib

artifacts_dir = Path(".") / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)

artifact_path = artifacts_dir / f"delay_model.joblib"
if best_model_name == "TwoStage":
    artifact = {
        "classifier": two_stage_clf,
        "regressor": two_stage_reg,
        "threshold": two_stage_threshold,
    }
    joblib.dump(artifact, artifact_path)
else:
    joblib.dump(best_pipe, artifact_path)

print("Artifact saved to:", artifact_path)

Artifact saved to: artifacts/delay_model.joblib


## Optuna tuning with TimeSeriesSplit

In [14]:
try:
    import optuna

    def objective(trial):
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 9),
            "max_iter": trial.suggest_int("max_iter", 100, 500),
            "random_state": 42,
        }
        
        tscv = TimeSeriesSplit(n_splits=3)
        mae_scores = []
        
        X_trainval_sorted = X_trainval.sort_index()
        y_trainval_sorted = y_trainval.sort_index()
        
        for train_idx, val_idx in tscv.split(X_trainval_sorted):
            X_tr, y_tr = X_trainval_sorted.iloc[train_idx], y_trainval_sorted.iloc[train_idx]
            X_va, y_va = X_trainval_sorted.iloc[val_idx], y_trainval_sorted.iloc[val_idx]
            
            # Using HistGradientBoostingRegressor for simplicity in tuning
            base_model = HistGradientBoostingRegressor(**params)
            model = get_log_regressor(base_model)
            
            pipe = Pipeline([
                ("preprocess", preprocess),
                ("model", model),
            ])
            
            pipe.fit(X_tr, y_tr)
            pred = pipe.predict(X_va)
            mae_scores.append(mean_absolute_error(y_va, pred))
            
        return np.mean(mae_scores)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=10) # 10 trials for demonstration

    print("Best parameters:", study.best_params)
except Exception as exc:
    print(f"Optuna not available or failed: {exc}")

[I 2026-05-09 09:54:12,527] A new study created in memory with name: no-name-848cae18-3e31-432e-b9ed-e7124f05b0fc
[I 2026-05-09 09:54:21,112] Trial 0 finished with value: 1.111045240549929 and parameters: {'learning_rate': 0.034596894210650396, 'max_depth': 7, 'max_iter': 122}. Best is trial 0 with value: 1.111045240549929.
[I 2026-05-09 09:54:48,059] Trial 1 finished with value: 1.1442538654588044 and parameters: {'learning_rate': 0.037273886336621986, 'max_depth': 7, 'max_iter': 394}. Best is trial 0 with value: 1.111045240549929.
[I 2026-05-09 09:54:54,947] Trial 2 finished with value: 1.109492459194973 and parameters: {'learning_rate': 0.012507943265282987, 'max_depth': 5, 'max_iter': 153}. Best is trial 2 with value: 1.109492459194973.
[I 2026-05-09 09:55:18,400] Trial 3 finished with value: 1.1131432306209332 and parameters: {'learning_rate': 0.01735392387667567, 'max_depth': 8, 'max_iter': 293}. Best is trial 2 with value: 1.109492459194973.
[I 2026-05-09 09:55:22,364] Trial 4 f

Best parameters: {'learning_rate': 0.016148930918754058, 'max_depth': 3, 'max_iter': 168}
